## CTG: avaliação final no conjunto de teste

- Pergunta que estamos buscando pela resposta: Os modelos escolhidos na validação cruzada separam bem exames normais, suspeitos e patológicos em exames que nunca viram? Quantos patológicos eles "erram"(pior caso)?



#### 0. Configuração inicial

In [1]:
import json
import subprocess
import warnings

import matplotlib.pyplot as plt
import numpy as np
from statstech.discriminant_analysis import (
    LinearDiscriminantAnalysis,
    QuadraticDiscriminantAnalysis,
)
from statstech.linear_model import LogisticRegression
from statstech.metrics.classification import confusion_matrix, roc_auc_score
from statstech.model_selection import bootstrap_ci
from statstech.neighbors import KNeighborsClassifier
from statstech.pipeline import Pipeline
from statstech.preprocessing import StandardScaler

from analysis.config import SEED, STYLE, TABLES
from analysis.evaluation import evaluate_classifier
from analysis.io import load_dataset
from analysis.report import save_figure, save_table

plt.style.use(STYLE)

warnings.filterwarnings("ignore", message=".*indefinid.*")

CLASS_NAMES = ["normal", "suspeito", "patológico"]
PATHOLOGICAL = 2
LABELS = {"logistic": "Logística", "lda": "LDA", "qda": "QDA", "knn": "KNN"}

#### 1. Dados 

In [2]:
data = load_dataset("ctg")
X_train, y_train = data["X_train"], data["y_train"]
X_test, y_test = data["X_test"], data["y_test"]

for name, y in [("treino", y_train), ("teste", y_test)]:
    counts = np.bincount(y, minlength=3)
    print(f"{name:6s} {len(y):5d} exames   " + "   ".join(
        f"{c}: {n} ({n / len(y):.1%})" for c, n in zip(CLASS_NAMES, counts)))

treino  1755 exames   normal: 1363 (77.7%)   suspeito: 245 (14.0%)   patológico: 147 (8.4%)
teste    359 exames   normal: 285 (79.4%)   suspeito: 47 (13.1%)   patológico: 27 (7.5%)


#### 2. Decisões antes do teste

##### 2.1 Modelos e Hiperparâmetros

Os hiperparâmetros escolhidos na comparação por validação cruzada estão em `ctg_hiperparametros.json`. Cada modelo é reconstruído a partir do arquivo ajustado no treino. 

In [3]:
with open(TABLES / "ctg_hiperparametros.json", encoding="utf-8") as f:
    best = json.load(f)

MODELS = {
    "logistic": LogisticRegression(),
    "lda": LinearDiscriminantAnalysis(),
    "qda": QuadraticDiscriminantAnalysis(),
    "knn": KNeighborsClassifier(),
}
finalists = {}
for key, model in MODELS.items():
    choice = best[key]
    # A transformação log1p não entrou em nenhum modelo final (ver ctg_models.ipynb).
    assert not choice["uses_log1p"]
    pipeline = Pipeline([("scaler", StandardScaler()), ("model", model)])
    pipeline.set_params(**choice["fixed"], **choice["params"])
    finalists[key] = pipeline.fit(X_train, y_train)
    print(f"{LABELS[key]:10s} {choice['params']}   F1 macro na validação cruzada: "
          f"{choice['f1_macro_cv']:.3f}")

Logística  {'model__alpha': 10}   F1 macro na validação cruzada: 0.780
LDA        {'model__reg_param': 1e-06}   F1 macro na validação cruzada: 0.715
QDA        {'model__reg_param': 0.5}   F1 macro na validação cruzada: 0.733
KNN        {'model__n_neighbors': 15}   F1 macro na validação cruzada: 0.723


##### 2.2 Modelo escolhido

É o modelo de maior F1 macro na validação cruzada.

In [4]:
chosen = max(MODELS, key=lambda key: best[key]["f1_macro_cv"])
print("modelo escolhido:", LABELS[chosen])

modelo escolhido: Logística


##### 2.3 Limiar de decisão 

Deixar passar um exame patológico é muito pior que um alarme falso. Logo, o modelo escolhido marca `patológico` sempre que a probabilidade dessa classe passa de um limiar `t`que foi escolhido no treino por validação cruzada. 

In [ ]:
# TODO implementar após a finalização da tarefa T6

#### 3. Teste 

`X_test` e `y_test` 